# wordle-0.2b — training

A ~214M-parameter transformer ("0.2B") that learns to play Wordle. Pure imitation learning: an entropy solver plays hundreds of thousands of games, and the model learns to predict the solver's next guess from the game state.

A GPU makes this fast (about 30 minutes on a free Colab/Kaggle T4). A CPU can run it too, just slower. Flip `SMOKE` to `True` below to check the pipeline in ~2 minutes on a tiny model.

In [ ]:
# one-time setup for Colab/Kaggle (uncomment if imports fail)
# %pip install -q -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__)

## 1. Config

`SMOKE=True` trains a ~7M-param toy to prove the pipeline end to end on CPU. `SMOKE=False` trains the real 0.2B model.

In [ ]:
SMOKE = False  # True = tiny CPU run to check the pipeline

if SMOKE:
    DATA_GAMES = 20_000
    STEPS, BATCH, LR = 600, 128, 3e-4
    N_LAYER, N_EMBD, N_HEAD = 4, 256, 4
    EVAL_EVERY, EVAL_GAMES = 150, 200
    RUN_NAME = "wordle-smoke"
else:
    DATA_GAMES = 300_000
    STEPS, BATCH, LR = 2_500, 512, 3e-4
    N_LAYER, N_EMBD, N_HEAD = 16, 1024, 16   # ~0.2B parameters
    EVAL_EVERY, EVAL_GAMES = 250, 200
    RUN_NAME = "wordle-0.2b"

SEED, WARMUP = 0, 200

## 2. Vocabulary and the feedback matrix

The corpus has 12,972 legal guess words, 2,315 of which can be answers. The pattern matrix maps every (guess, answer) pair to its feedback pattern; it's what makes the entropy solver fast, and it gets cached to `data/cache/` after the first build (~20s).

In [ ]:
from wordle02b import Vocabulary, load_word_lists, build_pattern_matrix_cached

answers, allowed = load_word_lists()
vocab = Vocabulary(answers, allowed)
P = build_pattern_matrix_cached(vocab, cache_dir="data/cache")

print(f"{vocab.V} words, {len(vocab.answers)} answers, vocab size {vocab.VOCAB_SIZE}")
print(f"pattern matrix: {P.shape}")

# 200 answers are held out: they never appear as secrets in training, so
# solving them later is honest generalization instead of memorization.
from wordle02b.evaluate import split_answers
train_ids, holdout_ids = split_answers(vocab, holdout_size=200, seed=42)
print(f"train secrets: {len(train_ids)}  holdout secrets: {len(holdout_ids)}")

## 3. The teacher

The baseline plays greedily: each move it picks the word that maximizes the expected information of the feedback, then filters candidates by what it actually saw. It wins ~99% of games in ~3.6 guesses. That is the ceiling for pure imitation, and the model's target.

In [ ]:
from wordle02b.evaluate import play_games_baseline, summarize

secrets_eval = [vocab.word(int(wid)) for wid in np.random.default_rng(42).choice(holdout_ids, 200, replace=False)]
st = summarize(play_games_baseline(vocab, P, 200, secrets_eval, seed=42))
print(f"teacher: solve {st['solve_rate']:.3f}  avg {st['avg_guesses']:.2f} guesses")

## 4. Training data

Games are generated with exploration (top-3 entropy guesses, 30% random openers) so the model sees many different game states instead of memorizing one trajectory per answer. Secrets come only from the train split; the 200 holdout answers never appear. Data is cached in `data/cache/`, so re-running with the same settings is instant.

300k games is roughly 7M tokens. On a laptop CPU generation takes about an hour on 8 cores; on Colab/Kaggle it's faster. The cache in `data/cache/` means you only pay once — to generate data separately: `python scripts/generate_data.py --games 300000 --workers 16`.

In [ ]:
from wordle02b.data import get_or_generate

tokens = get_or_generate(DATA_GAMES, vocab, P, cache_dir="data/cache", seed=SEED, secret_ids=train_ids)
print(f"{len(tokens):,} tokens")

## 5. The model

A compact GPT: 16 layers, 1024 hidden, 16 heads. Word-level vocabulary — every guess is one token, feedback letters are tokens too. Weight-tied embeddings, RMSNorm, causal attention. ~214M parameters, right at the 0.2B mark.

In [ ]:
from wordle02b.model import GPTConfig

cfg = GPTConfig(vocab_size=vocab.VOCAB_SIZE, block_size=40,
                n_layer=N_LAYER, n_embd=N_EMBD, n_head=N_HEAD)
print(f"{cfg.param_count() / 1e6:.1f}M parameters")

## 6. Train

Cross-entropy is computed only at word positions — the model is only ever asked to produce guesses, never feedback. Cosine LR schedule with warmup, gradient clipping. Every `EVAL_EVERY` steps the model plays fresh holdout games with greedy decoding (guesses that contradict known feedback are masked out — they can never be correct, so the filter is free), and you watch the solve rate climb in real time.

In [ ]:
from wordle02b.train import train
from wordle02b.evaluate import play_games

def eval_fn(step, model):
    games = play_games(model, vocab, EVAL_GAMES, secrets_eval, device=device)
    s = summarize(games)
    return {"solve": s["solve_rate"], "avg": s["avg_guesses"]}

model, history = train(
    tokens, cfg, device=device, out_dir="checkpoints",
    steps=STEPS, batch_size=BATCH, lr=LR, warmup_steps=WARMUP,
    eval_every=EVAL_EVERY, eval_games=EVAL_GAMES, eval_fn=eval_fn,
    word_count=vocab.V, seed=SEED, run_name=RUN_NAME,
)

## 7. Curves

In [ ]:
losses = [(h["step"], h["loss"]) for h in history if "loss" in h]
evals = [(h["step"], h["solve"]) for h in history if "solve" in h]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(*zip(*losses)); axes[0].set_title("training loss"); axes[0].set_xlabel("step")
if evals:
    axes[1].plot(*zip(*evals)); axes[1].set_title("eval solve rate"); axes[1].set_xlabel("step"); axes[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 8. What you have

- `checkpoints/wordle-0.2b-final.pt` — the trained model
- `notebooks/02_play_and_evaluate.ipynb` — play against it, watch it play
- `notebooks/03_benchmark_api_models.ipynb` — how it stacks up against GPT / Claude / DeepSeek
- `docs/improvement_walkthrough.md` — how to push it past the teacher's ceiling

Imitation caps you at the teacher's strength (~99%). Beating that means RL or search, and the walkthrough has the recipe.